# MA719 Project

## Fetch live data from the internet

I fetch two different kind of datasets:
1. all the german cities with data available -> this is saved with some additional information into `all_cities.json`
2. insolvencies information on all previously fetched cities. This list of json files is saved into `insolvencies.json`

Additionally I implemented some error handling for each fetch.
If nothing could be retrieved the program tries to read an old state from the filesystem.
On success this data is used for the rest of the program, on fail however the program terminates.

In [4]:
import os
import requests
import json
from pathlib import Path

ROOT_URL = "https://infranode.dev/api/v1/cities"
CITIES_FILE_PATH = "all_cities.json"
INSOLVENCIES_FILE_PATH = "insolvencies.json"

def fetch_data_from_url(url: str, _params: str = "") -> json:
    """Tries to fetch data from the given URL"""
    """Returns None on fails"""
    try: 
        response = requests.get(url, params=_params, timeout=10)
        response.raise_for_status()
        print(f"Successfully fetched live data from: {url}")
        return response.json()
    except Exception as e:
        print("Could not reach the service")
        return None

def readJsonFromFile(filepath: str) -> json:
    if (not Path(filepath).exists):
        print("File does not exist")
        return None

    with open(filepath, "r") as f:
        return json.load(f)

def saveJsonToFile(filepath: str, data: json) -> None:
    """Saves json datatype to given file"""
    if not Path(filepath).exists:
        print("File does not exist")
        return 
    
    with(open(filepath, "w")) as f:
        json.dump(data, f, indent=2)


all_cities = fetch_data_from_url(ROOT_URL)
if all_cities == None:
    print("Failed to retrieve cities Data - trying backup file")
    all_cities = readJsonFromFile(CITIES_FILE_PATH)
    if (all_cities == None):
        print("Failed to read all_cites data from backup file")
        exit(1)
else:
    saveJsonToFile(CITIES_FILE_PATH, all_cities)

insolvencies = []
for city in all_cities["data"]:
    slug = city.get("slug")
    url = ROOT_URL+"/"+slug+"/insolvencies"
    data = fetch_data_from_url(url)
    if (data == None):
        print(f"Failed to fetch data from: {url}")
        print("Continuing anyway")
        continue # skip append
        
    insolvencies.append(data)
print(f"Collected data from {len(insolvencies)}/{len(all_cities["data"])} cities")

if insolvencies == None:
    print("Failed to retrieve insolvencies data - trying backup file")
    insolvencies = readJsonFromFile(INSOLVENCIES_FILE_PATH)
    if insolvencies == None:
        print("Failed to read insolvencies from backup file")
        exit(1)
else: 
    saveJsonToFile(INSOLVENCIES_FILE_PATH, insolvencies)


Successfully fetched live data from: https://infranode.dev/api/v1/cities
Successfully fetched live data from: https://infranode.dev/api/v1/cities/berlin/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/hamburg/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/muenchen/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/koeln/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/frankfurt-am-main/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/stuttgart/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/duesseldorf/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/leipzig/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/dortmund/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/

## Build pandas data frame 

I loop through my collected data and filter the relevant info into a pandas data frame.
This data frame will be the data structure used by the analysis.

The data frame is structured like this:
City Name, Federal State, corporate insolvencies, other insolvencies, year

In [ ]:
import pandas as pd

# read_json only works for flat lists. The provided json files are deeply nested so I had to use a different command
df = pd.json_normalize(all_cities["data"])

df.head()

df = pd.json_normalize(insolvencies)

df.head()

ValueError: Mixing dicts with non-Series may lead to ambiguous ordering.